# KIT RAG（sarashina2.2-3b）Colab実行用
ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選んでから、上から順に実行。

In [ ]:
# 1. kit_rag.zip をアップロードして展開
from google.colab import files
files.upload()
!unzip -o -q kit_rag.zip
%cd kit_rag
!pip install -q -r requirements.txt

In [ ]:
# 2.（任意）Google Drive にデータを保存して、クロールのやり直しを防ぐ
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/kit_rag_data
# 既に保存済みなら復元:  !cp -r /content/drive/MyDrive/kit_rag_data/* data/

In [ ]:
# 3. ① 収集（1秒間隔なので数十分）
!python crawl.py

In [ ]:
# 4. ②③ 前処理・チャンク化 → ④ ベクトル索引
!python preprocess.py
!python index.py
!cp -r data/* /content/drive/MyDrive/kit_rag_data/ 2>/dev/null || true

In [ ]:
# 5. モデルを1回だけ読み込んで、自作の質問で確認
from pipeline import RAGPipeline
rag = RAGPipeline()
for q in ['図書館は何号館にありますか？',
          'コンビニは何号館の何階にありますか？',
          '夢考房とはどのような施設ですか？']:
    r = rag.answer(q)
    print(f"[{r['qtype']}] {q}\n  → {r['answer']}  (top_score={r['top_score']:.3f})\n  sources={r['sources'][:2]}\n")

In [ ]:
# 6. 検索だけを確認（回答が外れたとき、検索と生成のどちらが原因か切り分ける）
for chunk, score in rag.retriever.search('図書館は何号館にありますか？'):
    print(f'{score:.3f}', chunk.text[:80].replace('\n', ' '))

In [ ]:
# 7. 採点用の質問JSONで全問実行（形式は run_eval.py の TODO を合わせる）
import json, run_eval
qs = run_eval.load_questions('questions.json')
results = [{run_eval.ID_KEY: q.get(run_eval.ID_KEY, i), 'answer': rag.answer(q[run_eval.QUESTION_KEY])['answer']}
           for i, q in enumerate(qs)]
json.dump(results, open('answers.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
files.download('answers.json')